In [6]:
import os
from PIL import Image
from pathlib import Path
import shutil

def find_corrupted_images(root_dir):
    """Find all corrupted/truncated images in a directory tree"""
    corrupted = []
    
    for root, dirs, files in os.walk(root_dir):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png', '.gif')):
                img_path = os.path.join(root, file)
                try:
                    img = Image.open(img_path)
                    img.load()  # Force load to catch truncation
                except Exception as e:
                    corrupted.append({
                        'path': img_path,
                        'error': str(e)
                    })
    
    return corrupted

# Find corrupted images
corrupted_train_list = find_corrupted_images("butterfly_anomaly_image\\train")

print(f"Found {len(corrupted_train_list)} corrupted images:")

corrupted_test_list = find_corrupted_images("butterfly_anomaly_image\\test")

print(f"Found {len(corrupted_test_list)} corrupted images in test set:")

corrupted_val_list = find_corrupted_images("butterfly_anomaly_image\\val")

print(f"Found {len(corrupted_val_list)} corrupted images in validation set:")

destination_train = "corrupted_train_image"
destination_test = "corrupted_test_image"
destination_val = "corrupted_val_image"

for item in corrupted_train_list:
    shutil.move(item['path'], destination_train)

for item in corrupted_test_list:
    shutil.move(item['path'], destination_test)

for item in corrupted_val_list:
    shutil.move(item['path'], destination_val)



Found 6 corrupted images:
Found 1 corrupted images in test set:
Found 1 corrupted images in validation set:


In [ ]:
import os
import shutil
import random

def split_dataset(
    input_dir,
    output_dir,
    train_ratio=0.7,
    val_ratio=0.15,
    test_ratio=0.15,
    seed=42,
    dry_run=False
):
    random.seed(seed)

    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6

    splits = ['train', 'val', 'test']

    # Create output directories (only if not dry run)
    for split in splits:
        for class_name in os.listdir(input_dir):
            class_path = os.path.join(output_dir, split, class_name)
            if not dry_run:
                os.makedirs(class_path, exist_ok=True)
            else:
                print(f"[DRY RUN] Would create: {class_path}")

    for class_name in os.listdir(input_dir):
        class_input_path = os.path.join(input_dir, class_name)

        if not os.path.isdir(class_input_path):
            continue

        files = [
            f for f in os.listdir(class_input_path)
            if os.path.isfile(os.path.join(class_input_path, f))
        ]

        random.shuffle(files)

        total = len(files)
        train_end = int(total * train_ratio)
        val_end = train_end + int(total * val_ratio)

        split_map = {
            'train': files[:train_end],
            'val': files[train_end:val_end],
            'test': files[val_end:]
        }

        for split, file_list in split_map.items():
            for file in file_list:
                src = os.path.join(class_input_path, file)
                dst = os.path.join(output_dir, split, class_name, file)

                if dry_run:
                    print(f"[DRY RUN] {src} -> {dst}")
                else:
                    shutil.copy2(src, dst)

    print("Dry run complete!" if dry_run else "Dataset split complete!")

split_dataset("butterfly_anomaly_train_image", "butterfly_anomaly_image", dry_run=True)